In [1]:
! pip install -q lovely-tensors
import lovely_tensors as lt; lt.monkey_patch()

In [2]:
import sys; sys.path.append('..')

In [3]:

import torch
from torch.optim import Adam
from datasets import load_dataset
from typing import Optional
import asyncio
import pandas as pd
from datetime import datetime

from early_exit.util import get_model, load_model_from_wandb, load_model, configs_from_json, save_model
from early_exit.rl_utils import apply_masking, create_attention_mask_from_tokens, generate_k_completions, center_rewards_per_prompt, map_layers_to_indices, weighted_sft_step, get_input_prompt_length, evaluate_coherence, compute_sample_labels, load_gsm8k_with_difficulty, compute_accuracy_by_difficulty
from early_exit.rl_types import RLHyperparams, RolloutBatch
from early_exit.rewards import compute_verification_rewards, compute_token_kl_from_logprobs, compute_token_logprobs_reference, compute_token_logprobs_student, compute_avg_exit_layer, extract_solution
from early_exit.patching import replace_attention_layers, set_transformer_early_exit_mode
from shared_utils.load import get_tokenizer, configs_from_yaml
from torch.nn.utils.rnn import pad_sequence
from pathlib import Path


device = "cuda"
device = 'cpu'

ROOT_DIR = Path('..')

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
config_path = ROOT_DIR / "config_deepseek.yaml"
sft_model_path = ROOT_DIR / "models/early_exit_20250908_layers_5_big"  # TODO: set path to SFT checkpoint

RL_HPARAMS = RLHyperparams()
training_steps_per_rollout = 4

save_freq = 250
save_dir = f"models/rl_{datetime.now().strftime('%Y%m%d')}"

# --- Models (schema) ---
tokenizer = get_tokenizer(model_name)
config = configs_from_yaml(config_path, tokenizer.eos_token_id)

# student = get_model(model_name, config['model'], device)
# student = replace_attention_layers(student, config['lora'], device)
# # TODO: Change artifact path to sft trained gsm-8k model
# student = load_model_from_wandb(student, model_path = "models/sft_model_2", artifact_path = 'vkarthik095-university-of-amsterdam/early-exit/early_exit_20250908_layers_5_big:v0')
# #student = load_model(student, sft_model_path)

# # Reference policy: base unmodified model without early exit
# reference = get_model(model_name, config['model'], device)
# reference.eval()
# TODO: ensure no early-exit logic is active for reference model

# Dataset
dataset = load_gsm8k_with_difficulty()

/Users/mariiakoroliuk/miniconda3/envs/gradio_ext/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ---------------- RL helper functions moved from train_rl.py ----------------
def generate_k_completions_batched(
        model, 
        prompt, 
        k: int, 
        tokenizer, 
        config, 
        device, 
        system_prompt
    ):
    """
    Free-generate K completions per prompt with early exits enabled.

    Expected outputs (used later in the pipeline):
    - completions:
        - tokens: LongTensor of shape [batch*K, seq_len]; dtype=torch.long
        - texts: list[str] of length batch*K
    - exit_info:
        - prescribed_exit_layers: Optional[LongTensor] of shape [batch*K, seq_len] for re-scoring

    Typical ranges:
    - seq_len: 16–512 depending on generation configuration
    """
    set_transformer_early_exit_mode(model, 'free_generate')

    all_tokens = []
    all_texts = []
    all_prescribed_exit_layers = []
    
    # Batch all generations together: [p1, p1, ..., p1 (k times), p2, p2, ..., p2 (k times), ...]
    batched_prompts = [p for p in prompt for _ in range(k)]
    
    with torch.no_grad():
        decoded_responses, model_outputs = generate_text(
            model=model,
            prompt=batched_prompts,
            system_prompt=system_prompt,
            prefiller='',
            tokenizer=tokenizer,
            generation_config=config['generation'],
            device=device
        )
    
    sequences, exit_layer_idxs = model_outputs
    
    # Process all outputs from the batch
    for i in range(len(batched_prompts)):
        tokens = sequences[i]
        prescribed_exit_layers = exit_layer_idxs[i]
        
        all_tokens.append(tokens[:-1])
        all_texts.append(decoded_responses[i])
        all_prescribed_exit_layers.append(prescribed_exit_layers[1:])
    
    max_seq_len = max(len(tokens) for tokens in all_tokens)
    padded_tokens = []
    final_prescribed_layers = []  # no padding since will mess up avg exit layer

    for i, (tokens, exit_layers) in enumerate(zip(all_tokens, all_prescribed_exit_layers)):
        pad_length = max_seq_len - len(tokens)
        if pad_length > 0:
            padded_token = torch.cat([tokens, torch.full((pad_length,), tokenizer.pad_token_id, dtype=tokens.dtype, device=tokens.device)])
        else:
            padded_token = tokens

        padded_tokens.append(padded_token)
        final_prescribed_layers.append(exit_layers)

    completions_tokens = torch.stack(padded_tokens, dim=0)

    completions = {
        'tokens': completions_tokens,
        'texts': all_texts
    }

    exit_info = {
        'prescribed_exit_layers': final_prescribed_layers
    }
    
    return completions, exit_info


In [5]:
student = get_model(model_name, config['model'], device)
student = replace_attention_layers(student, config['lora'], device)
# # TODO: Change artifact path to sft trained gsm-8k model
student = load_model_from_wandb(student, model_path = sft_model_path, artifact_path = 'vkarthik095-university-of-amsterdam/early-exit/early_exit_20250908_layers_5_big:v0')
student = load_model(student, sft_model_path)

replacing generate_layer_type_without_early_exit_decision_head layer model.layers.0
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.1
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.2
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.3
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.4
replacing layer model.layers.5
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.6
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.7
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.8
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.9
replacing layer model.layers.10
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.11
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.12
replacing g

/Users/mariiakoroliuk/miniconda3/envs/gradio_ext/lib/python3.11/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


trainable params: 4,358,144 || all params: 1,805,069,829 || trainable%: 0.2414
Model path ../models/early_exit_20250908_layers_5_big already exists, skipping download


In [6]:
prompts = [
    "Who is the president of Burundi",
    "Анекдот розкажи"
]


generation_config = {**config['generation'], 'max_new_tokens': 4}
model = student

In [7]:
import torch
from torch.optim import Adam

from datasets import load_dataset, Dataset
import pandas as pd

from early_exit.patching import set_transformer_early_exit_mode
from shared_utils.generate import generate_text
from typing import List

from early_exit.rl_types import *

from shared_utils.generate import generate_text

In [8]:
#         # 1) Rollouts (student free-generate K)
# completions, exit_info = generate_k_completions(student, prompts, k=RL_HPARAMS.k, 
#                                                 tokenizer=tokenizer, config=config, device=device, 
#                                                 system_prompt = RL_HPARAMS.system_prompt)  # TODO

In [9]:
RL_HPARAMS.k

4

In [10]:
from shared_utils.generate import generate_text, format_conversation, full_tokenize

In [11]:
model = student
prompt = prompts
k = RL_HPARAMS.k
tokenizer = tokenizer
config = config
device = device
system_prompt = RL_HPARAMS.system_prompt
prefiller = None

all_tokens = []
all_texts = []
all_prescribed_exit_layers = []

# Batch all generations together: [p1, p1, ..., p1 (k times), p2, p2, ..., p2 (k times), ...]
batched_prompts = [p for p in prompt for _ in range(k)]

with torch.no_grad():
    pre_transformed_conversation = format_conversation(user_prompts = prompt, system_prompt=system_prompt)
    full_prompts = tokenizer.apply_chat_template(pre_transformed_conversation, 
                                                 prefiller=prefiller,
                                                 tokenize=False,
                                                 add_generation_prompt=True # adds the <｜Assistant｜> token at the end
                                                 )
    inputs = full_tokenize(prompts=full_prompts, tokenizer=tokenizer, device = device, add_special_tokens=False)
    print('prompt tokens shape:', inputs['input_ids'].shape)
    all_model_outputs = model.generate(**inputs, **generation_config)
    decoded_responses = tokenizer.batch_decode(all_model_outputs[0].squeeze())
    # decoded_responses = tokenizer.batch_decode(all_model_outputs, skip_special_tokens=True)  

full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([2, 44])


In [21]:
decoded_responses = tokenizer.batch_decode(all_model_outputs[0].squeeze())

In [16]:
all_model_outputs

(tensor[2, 48] i64 n=96 x∈[11, 151648] μ=2.436e+04 σ=4.800e+04,
 tensor[2, 4] n=8 μ=25.000 σ=0. +Inf! [[inf, 25.000, inf, inf], [inf, 25.000, inf, inf]])

In [13]:
        # 1) Rollouts (student free-generate K)
completions, exit_info = generate_k_completions_batched(student, prompts, k=RL_HPARAMS.k, 
                                                tokenizer=tokenizer, config=config, device=device, 
                                                system_prompt = RL_HPARAMS.system_prompt)  # TODO

full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([8, 44])


In [15]:
completions

{'tokens': tensor[8, 743] i64 n=5944 (46Kb) x∈[1, 151649] μ=2.886e+04 σ=5.495e+04,
 'texts': ["<｜end▁of▁sentence｜><｜end▁of▁sentence｜><｜begin▁of▁sentence｜>I am going to give you a math word problem. Solve it step by step, showing your reasoning. After your work, provide your final numerical answer<｜User｜>Who is the president of Burundi<｜Assistant｜><think>\nAlright, so I need to figure out which person is the president of burundi. First, I'll start by understanding what a president is. The president is the head of government in a country and is typically responsible for leading the country's political life, including economic policies and foreign relations.\n\nNow, I need to identify the president of Burundi. I'll think about the country's history and notable figures. I'll consider historical events, the government structure, and any significant roles held by individuals.\n\nI know that Burundi has a rich history dating back to ancient times. The language, Burundas, is related to both Fr

In [10]:
add_special_tokens = False

inputs = tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    add_special_tokens=add_special_tokens
).to(device)

inputs['input_ids'].shape

torch.Size([2, 9])

In [11]:
all_model_outputs = model.generate(**inputs, **generation_config)

In [12]:
# decoded_responses = tokenizer.decode(all_model_outputs)


for i in range(len(all_model_outputs)):
    print ( tokenizer.decode(all_model_outputs[i].squeeze()) )

<｜end▁of▁sentence｜><｜end▁of▁sentence｜>Who is the president of Burundi, and can they
Анекдот розкажи: There are 


## Debug Capture: Intercept patched_layer_forward inputs

This section captures all inputs to `patched_layer_forward` for debugging both scenarios:
- **Free Generate Mode**: `unfrozen_idx_or_mask` as a list
- **SFT Student Mode**: `unfrozen_idx_or_mask` as a tensor


In [52]:
student = get_model(model_name, config['model'], device)
student = replace_attention_layers(student, config['lora'], device)
# # TODO: Change artifact path to sft trained gsm-8k model
student = load_model_from_wandb(student, model_path = sft_model_path, artifact_path = 'vkarthik095-university-of-amsterdam/early-exit/early_exit_20250908_layers_5_big:v0')
student = load_model(student, sft_model_path)

replacing generate_layer_type_without_early_exit_decision_head layer model.layers.0
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.1
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.2
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.3
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.4
replacing layer model.layers.5
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.6
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.7
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.8
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.9
replacing layer model.layers.10
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.11
replacing generate_layer_type_without_early_exit_decision_head layer model.layers.12
replacing g

In [53]:
from types import MethodType

# Global storage for captured inputs
captured_inputs = {
    'scenario': None,  # 'free_generate' or 'sft_student'
    'hidden_states': None,
    'attention_mask': None,
    'position_ids': None,
    'past_key_value': None,
    'output_attentions': None,
    'use_cache': None,
    'cache_position': None,
    'position_embeddings': None,
    'unfrozen_idx_or_mask': None,
    'kwargs': None,
    'layer_idx': None,
}

# Store original method reference
original_patched_layer_forward = None

def create_interceptor(original_method, layer_idx, scenario_name):
    """
    Wrapper that intercepts all arguments to patched_layer_forward
    and stores them in the global captured_inputs dict.
    """
    def interceptor_wrapper(
        self,
        hidden_states,
        attention_mask=None,
        position_ids=None,
        past_key_value=None,
        output_attentions=False,
        use_cache=False,
        cache_position=None,
        position_embeddings=None,
        unfrozen_idx_or_mask=None,
        **kwargs
    ):
        global captured_inputs
        
        # Capture all inputs
        captured_inputs['scenario'] = scenario_name
        captured_inputs['hidden_states'] = hidden_states.clone() if hidden_states is not None else None
        captured_inputs['attention_mask'] = attention_mask.clone() if attention_mask is not None else None
        captured_inputs['position_ids'] = position_ids.clone() if position_ids is not None else None
        captured_inputs['past_key_value'] = past_key_value  # Cache object, don't clone
        captured_inputs['output_attentions'] = output_attentions
        captured_inputs['use_cache'] = use_cache
        captured_inputs['cache_position'] = cache_position.clone() if cache_position is not None else None
        captured_inputs['position_embeddings'] = (
            (position_embeddings[0].clone(), position_embeddings[1].clone()) 
            if position_embeddings is not None else None
        )
        captured_inputs['layer_idx'] = layer_idx
        captured_inputs['kwargs'] = kwargs.copy()
        
        # Special handling for unfrozen_idx_or_mask
        if isinstance(unfrozen_idx_or_mask, list):
            captured_inputs['unfrozen_idx_or_mask'] = unfrozen_idx_or_mask.copy()
        elif isinstance(unfrozen_idx_or_mask, torch.Tensor):
            captured_inputs['unfrozen_idx_or_mask'] = unfrozen_idx_or_mask.clone()
        else:
            captured_inputs['unfrozen_idx_or_mask'] = unfrozen_idx_or_mask
        
        print(f"🔍 Intercepted call to patched_layer_forward at layer {layer_idx}")
        print(f"   Scenario: {scenario_name}")
        print(f"   unfrozen_idx_or_mask type: {type(unfrozen_idx_or_mask)}")
        if isinstance(unfrozen_idx_or_mask, list):
            print(f"   unfrozen_idx_or_mask (list): {unfrozen_idx_or_mask}")
        elif isinstance(unfrozen_idx_or_mask, torch.Tensor):
            print(f"   unfrozen_idx_or_mask shape: {unfrozen_idx_or_mask.shape}")
        
        # Call original method
        return original_method(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_value=past_key_value,
            output_attentions=output_attentions,
            use_cache=use_cache,
            cache_position=cache_position,
            position_embeddings=position_embeddings,
            unfrozen_idx_or_mask=unfrozen_idx_or_mask,
            **kwargs
        )
    
    return interceptor_wrapper

print("✓ Interceptor setup ready")


✓ Interceptor setup ready


In [58]:
# Apply monkey patch to a specific layer (layer 5 as an example)
TARGET_LAYER_IDX = 5

# Access the target layer
target_layer = student.base_model.model.model.layers[TARGET_LAYER_IDX]

# Store original method
original_patched_layer_forward = target_layer.patched_layer_forward

print(f"✓ Target layer {TARGET_LAYER_IDX} accessed")
print(f"  Layer type: {type(target_layer)}")
print(f"  Has patched_layer_forward: {hasattr(target_layer, 'patched_layer_forward')}")


✓ Target layer 5 accessed
  Layer type: <class 'early_exit.patching.dynamical_types.generate_layer_type_with_early_exit_decision_head.<locals>.DynamicallyTypedLayerWithExit'>
  Has patched_layer_forward: True


### Scenario A: Free Generate Mode (unfrozen_idx_or_mask as list)


In [157]:
# Install interceptor for free_generate scenario (properly bound as instance method)
interceptor_func = create_interceptor(
    original_patched_layer_forward, 
    TARGET_LAYER_IDX, 
    'free_generate'
)
target_layer.patched_layer_forward = MethodType(interceptor_func, target_layer)

# Set mode
set_transformer_early_exit_mode(student, 'free_generate')

# Trigger generation
test_prompts = [
    "Who is the president of Burundi",
    # "Who is the president of Rwanda",
    # "Харцизьк - це місто в",
]



with torch.no_grad():
    decoded_response, model_outputs = generate_text(
        model=student,
        prompt=test_prompts,
        system_prompt=RL_HPARAMS.system_prompt,
        prefiller='',
        tokenizer=tokenizer,
        generation_config={**config['generation'], 'max_new_tokens': 4},
        device=device
    )

print("\n" + "="*60)
print("FREE GENERATE MODE CAPTURE COMPLETE")
print("="*60)

full_tokenize currently only for Deepseek models!
prompt tokens shape: torch.Size([1, 42])
🔍 Intercepted call to patched_layer_forward at layer 0
   Scenario: free_generate
   unfrozen_idx_or_mask type: <class 'NoneType'>
🔍 Intercepted call to patched_layer_forward at layer 5
   Scenario: free_generate
   unfrozen_idx_or_mask type: <class 'NoneType'>
🔍 Intercepted call to patched_layer_forward at layer 0
   Scenario: free_generate
   unfrozen_idx_or_mask type: <class 'list'>
   unfrozen_idx_or_mask (list): [0]
🔍 Intercepted call to patched_layer_forward at layer 5
   Scenario: free_generate
   unfrozen_idx_or_mask type: <class 'list'>
   unfrozen_idx_or_mask (list): [0]
🔍 Intercepted call to patched_layer_forward at layer 0
   Scenario: free_generate
   unfrozen_idx_or_mask type: <class 'list'>
   unfrozen_idx_or_mask (list): [0]
🔍 Intercepted call to patched_layer_forward at layer 5
   Scenario: free_generate
   unfrozen_idx_or_mask type: <class 'list'>
   unfrozen_idx_or_mask (list):

TypeError: argument 'ids': 'list' object cannot be interpreted as an integer

In [60]:
isinstance(captured_inputs['hidden_states'], torch.Tensor)

True

In [158]:
from copy import deepcopy

def deep_tensor_copy(dict_to_copy):
    new_dict = {}
    for key, value in dict_to_copy.items():
        if isinstance(value, torch.Tensor):
            new_dict[key] = value.clone()
        else:
            new_dict[key] = deepcopy(value)
    return new_dict

# captured_inputs_bs1 = deepcopy(captured_inputs)
captured_inputs_bs3 = deep_tensor_copy(captured_inputs)

In [159]:
from transformers.utils import logging
logger = logging.get_logger(__name__)

from early_exit.patching.attention_mixins.base import LayerFakeAttentionForwardMixin

import torch
from torch.nn import functional as F
from torch import nn, Tensor as _T, FloatTensor as _FT, LongTensor as _LT

import math

from typing import List, Optional, Tuple

from transformers import Cache
from transformers.models.qwen2.modeling_qwen2 import apply_rotary_pos_emb, repeat_kv, Qwen2Attention

In [160]:
captured_inputs = captured_inputs_bs3
captured_inputs['use_cache'] = False

hidden_states = captured_inputs['hidden_states']
attention_mask = captured_inputs['attention_mask']
position_ids = captured_inputs['position_ids']
past_key_value = None # captured_inputs['past_key_value']
output_attentions = captured_inputs['output_attentions']
use_cache = captured_inputs['use_cache']
cache_position = captured_inputs['cache_position']
position_embeddings = captured_inputs['position_embeddings']
unfrozen_idx_or_mask = captured_inputs['unfrozen_idx_or_mask']

self = target_layer


captured_inputs

{'scenario': 'free_generate',
 'hidden_states': tensor[1, 1, 1536] 6Kb x∈[-12.828, 16.890] μ=0.071 σ=1.967,
 'attention_mask': tensor[1, 1, 1, 45] all_zeros,
 'position_ids': tensor[1, 1] i64 [[44]],
 'past_key_value': <transformers.cache_utils.DynamicCache at 0x1e4273190>,
 'output_attentions': False,
 'use_cache': False,
 'cache_position': tensor[1] i64 [44],
 'position_embeddings': (tensor[1, 1, 128] x∈[-0.988, 1.000] μ=0.542 σ=0.679,
  tensor[1, 1, 128] x∈[-0.952, 1.000] μ=0.186 σ=0.466),
 'unfrozen_idx_or_mask': [0],
 'kwargs': {},
 'layer_idx': 5}

In [161]:
_original_hidden_states = hidden_states.clone()

bsz, q_len, _ = hidden_states.size()

if isinstance(unfrozen_idx_or_mask, list):
    # unfrozen_elements = unfrozen_idx_or_mask
    unfrozen_mask = torch.zeros(bsz, dtype=torch.bool, device=hidden_states.device)
    if len(unfrozen_idx_or_mask) > 0:
        unfrozen_mask[unfrozen_idx_or_mask] = True
    unfrozen_elements = unfrozen_mask
    
elif isinstance(unfrozen_idx_or_mask, _T):
    # XXX: CHECK MASK AND ATTENTION ALIGNMENT BY TIME
    gen_len = unfrozen_idx_or_mask.shape[1]
    padding_required = q_len - gen_len
    unfrozen_elements = F.pad(
        input = unfrozen_idx_or_mask,
        pad = (padding_required, 0),
        value = True    # Pre-rollout (prompt) residual stream never gets frozen
    ).to(hidden_states.device)

elif unfrozen_idx_or_mask is None:
    # unfrozen_elements = torch.arange(bsz)
    unfrozen_elements = torch.ones(bsz, dtype=torch.bool, device=hidden_states.device)

residual = hidden_states.clone()

# hidden_states[unfrozen_elements] = self.input_layernorm(hidden_states[unfrozen_elements])
normed_hidden_states = self.input_layernorm(hidden_states)
# Self Attention
attn_result = self.self_attn(
    hidden_states=normed_hidden_states,
    attention_mask=attention_mask,
    position_ids=position_ids,
    past_key_value=past_key_value,
    output_attentions=output_attentions,
    use_cache=use_cache,
    cache_position=cache_position,
    position_embeddings=position_embeddings,
    unfrozen_idx_or_mask=unfrozen_idx_or_mask       # Key change
)

if len(attn_result) == 2:
    hidden_states, self_attn_weights = attn_result
    present_key_value = None
elif len(attn_result) == 3:
    hidden_states, self_attn_weights, present_key_value = attn_result
else:
    raise ValueError(f"Unexpected attention return format: {len(attn_result)} values")


print(f'hidden_states: {hidden_states.shape} {unfrozen_elements.unsqueeze(-1).shape=} {residual.shape=}')
hidden_states = torch.where(unfrozen_elements.unsqueeze(-1).unsqueeze(-1), residual + hidden_states, residual)
print(f'hidden_states: {hidden_states}')

# Fully Connected
residual = hidden_states.clone()
hidden_states[unfrozen_elements] = self.post_attention_layernorm(hidden_states[unfrozen_elements])
hidden_states[unfrozen_elements] = self.mlp(hidden_states[unfrozen_elements])
hidden_states[unfrozen_elements] = residual[unfrozen_elements] + hidden_states[unfrozen_elements]

outputs = (hidden_states,)

if output_attentions:
    outputs += (self_attn_weights,)

if use_cache:
    outputs += (present_key_value,)

assert (_original_hidden_states == hidden_states)[~unfrozen_elements].all()

hidden_states: torch.Size([1, 1, 1536]) unfrozen_elements.unsqueeze(-1).shape=torch.Size([1, 1]) residual.shape=torch.Size([1, 1, 1536])
hidden_states: tensor[1, 1, 1536] 6Kb x∈[-22.087, 15.979] μ=0.061 σ=2.510 grad WhereBackward0


In [155]:
hidden_states

tensor[3, 48, 1536] n=221184 (0.8Mb) x∈[-16.802, 14.566] μ=-0.043 σ=2.364 grad AddBackward0

tensor[3] bool x∈[True, True] μ=1.000 σ=0. [True, True, True]

In [150]:
_original_hidden_states = hidden_states.clone()

bsz, q_len, _ = hidden_states.size()

if isinstance(unfrozen_idx_or_mask, list):
    # unfrozen_elements = unfrozen_idx_or_mask
    unfrozen_mask = torch.zeros(bsz, dtype=torch.bool, device=hidden_states.device)
    if len(unfrozen_idx_or_mask) > 0:
        unfrozen_mask[unfrozen_idx_or_mask] = True
    unfrozen_elements = unfrozen_mask
    
elif isinstance(unfrozen_idx_or_mask, _T):
    # XXX: CHECK MASK AND ATTENTION ALIGNMENT BY TIME
    gen_len = unfrozen_idx_or_mask.shape[1]
    padding_required = q_len - gen_len
    unfrozen_elements = F.pad(
        input = unfrozen_idx_or_mask,
        pad = (padding_required, 0),
        value = True    # Pre-rollout (prompt) residual stream never gets frozen
    ).to(hidden_states.device)

elif unfrozen_idx_or_mask is None:
    # unfrozen_elements = torch.arange(bsz)
    unfrozen_elements = torch.ones(bsz, dtype=torch.bool, device=hidden_states.device)

residual = hidden_states.clone()

# hidden_states[unfrozen_elements] = self.input_layernorm(hidden_states[unfrozen_elements])
normed_hidden_states = self.input_layernorm(hidden_states)
# Self Attention
attn_result = self.self_attn(
    hidden_states=normed_hidden_states,
    attention_mask=attention_mask,
    position_ids=position_ids,
    past_key_value=past_key_value,
    output_attentions=output_attentions,
    use_cache=use_cache,
    cache_position=cache_position,
    position_embeddings=position_embeddings,
    unfrozen_idx_or_mask=unfrozen_idx_or_mask       # Key change
)

if len(attn_result) == 2:
    hidden_states, self_attn_weights = attn_result
    present_key_value = None
elif len(attn_result) == 3:
    hidden_states, self_attn_weights, present_key_value = attn_result
else:
    raise ValueError(f"Unexpected attention return format: {len(attn_result)} values")

print(f'hidden_states: {hidden_states.shape} {unfrozen_elements.unsqueeze(-1).shape=} {residual.shape=}')
print(torch.where(unfrozen_elements.unsqueeze(-1).unsqueeze(-1), residual + hidden_states, residual))
hidden_states = torch.where(unfrozen_elements.unsqueeze(-1), residual + hidden_states, residual)
print(f'hidden_states: {hidden_states}')

# Fully Connected
residual = hidden_states.clone()
hidden_states[unfrozen_elements] = self.post_attention_layernorm(hidden_states[unfrozen_elements])
hidden_states[unfrozen_elements] = self.mlp(hidden_states[unfrozen_elements])
hidden_states[unfrozen_elements] = residual[unfrozen_elements] + hidden_states[unfrozen_elements]

outputs = (hidden_states,)

if output_attentions:
    outputs += (self_attn_weights,)

if use_cache:
    outputs += (present_key_value,)

assert (_original_hidden_states == hidden_states)[~unfrozen_elements].all()

hidden_states: torch.Size([3, 48, 1536]) unfrozen_elements.unsqueeze(-1).shape=torch.Size([3, 1]) residual.shape=torch.Size([3, 48, 1536])
tensor[3, 48, 1536] n=221184 (0.8Mb) x∈[-19.859, 20.944] μ=0.027 σ=2.616 grad WhereBackward0
hidden_states: tensor[3, 48, 1536] n=221184 (0.8Mb) x∈[-19.859, 20.944] μ=0.027 σ=2.616 grad WhereBackward0


tensor[3, 3, 48, 1536] n=663552 (2.5Mb) x∈[-96.922, 114.626] μ=-0.315 σ=28.747 grad IndexPutBackward0

In [ ]:
# hidden_states: torch.Size([3, 48, 1536]) unfrozen_elements.unsqueeze(-1).shape=torch.Size([3, 1]) residual.shape=torch.Size([3, 48, 1536])
# hidden_states: torch.Size([3, 48, 1536]) unfrozen_elements.unsqueeze(-1).shape=torch.Size([3, 1]) residual.shape=torch.Size([3, 48, 1536])


tensor[3] bool x∈[True, True] μ=1.000 σ=0. [True, True, True]

In [ ]:
hidden_states = torch.where(unfrozen_elements.unsqueeze(-1), residual + hidden_states, residual)

RuntimeError: The size of tensor a (3) must match the size of tensor b (48) at non-singleton dimension 1

In [43]:
dict(
    
    hidden_states=hidden_states,
    attention_mask=attention_mask,
    position_ids=position_ids,
    past_key_value=past_key_value,
    output_attentions=output_attentions,
    use_cache=use_cache,
    cache_position=cache_position,
    position_embeddings=position_embeddings,
    unfrozen_idx_or_mask=unfrozen_idx_or_mask       # Key change
)


{'hidden_states': tensor[3, 48, 1536] n=221184 (0.8Mb) x∈[-0.181, 0.197] μ=0.000 σ=0.031,
 'attention_mask': tensor[3, 1, 48, 48] n=6912 (27Kb) x∈[-3.403e+38, 0.] μ=-inf σ=inf,
 'position_ids': tensor[1, 48] i64 x∈[0, 47] μ=23.500 σ=14.000,
 'past_key_value': <transformers.cache_utils.DynamicCache at 0x1a008ff90>,
 'output_attentions': False,
 'use_cache': True,
 'cache_position': tensor[48] i64 x∈[0, 47] μ=23.500 σ=14.000,
 'position_embeddings': (tensor[1, 48, 128] n=6144 (24Kb) x∈[-1.000, 1.000] μ=0.622 σ=0.622,
  tensor[1, 48, 128] n=6144 (24Kb) x∈[-1.000, 1.000] μ=0.167 σ=0.445),
 'unfrozen_idx_or_mask': None}

In [30]:
dict(
    
    hidden_states=normed_hidden_states,
    attention_mask=attention_mask,
    position_ids=position_ids,
    past_key_value=past_key_value,
    output_attentions=output_attentions,
    use_cache=use_cache,
    cache_position=cache_position,
    position_embeddings=position_embeddings,
    unfrozen_idx_or_mask=unfrozen_idx_or_mask       # Key change
)


{'hidden_states': tensor[3, 48, 1536] n=221184 (0.8Mb) x∈[-5.311, 5.687] μ=0.002 σ=0.294,
 'attention_mask': tensor[3, 1, 48, 48] n=6912 (27Kb) x∈[-3.403e+38, 0.] μ=-inf σ=inf,
 'position_ids': tensor[1, 48] i64 x∈[0, 47] μ=23.500 σ=14.000,
 'past_key_value': <transformers.cache_utils.DynamicCache at 0x1a009e2d0>,
 'output_attentions': False,
 'use_cache': True,
 'cache_position': tensor[48] i64 x∈[0, 47] μ=23.500 σ=14.000,
 'position_embeddings': (tensor[1, 48, 128] n=6144 (24Kb) x∈[-1.000, 1.000] μ=0.622 σ=0.622,
  tensor[1, 48, 128] n=6144 (24Kb) x∈[-1.000, 1.000] μ=0.167 σ=0.445),
 'unfrozen_idx_or_mask': None}

### Scenario B: SFT Student Mode (unfrozen_idx_or_mask as tensor)


In [ ]:
# Install interceptor for sft_student scenario (properly bound as instance method)
interceptor_func = create_interceptor(
    original_patched_layer_forward, 
    TARGET_LAYER_IDX, 
    'sft_student'
)
target_layer.patched_layer_forward = MethodType(interceptor_func, target_layer)

# Set mode
set_transformer_early_exit_mode(student, 'sft_student')

# Prepare inputs for SFT student mode
batch_size = 2
prompt_length = 20
generation_length = 10
total_length = prompt_length + generation_length

# Create mock input_ids
mock_input_ids = torch.randint(0, 1000, (batch_size, total_length), device=device)

# Create prescribed_exit_layer_idxs (shape: [batch, generation_length])
# Values represent which layer each token should exit at
# Example: layer indices 0-5, with some early exits
prescribed_exit_layer_idxs = torch.randint(0, 6, (batch_size, generation_length), device=device)

print(f"Mock input setup:")
print(f"  input_ids shape: {mock_input_ids.shape}")
print(f"  prescribed_exit_layer_idxs shape: {prescribed_exit_layer_idxs.shape}")
print(f"  prescribed exits: {prescribed_exit_layer_idxs}")

# Trigger forward pass
with torch.no_grad():
    outputs, exit_logits = student(
        input_ids=mock_input_ids,
        prescribed_exit_layer_idxs=prescribed_exit_layer_idxs
    )

print("\n" + "="*60)
print("SFT STUDENT MODE CAPTURE COMPLETE")
print("="*60)


### Inspection Helpers


In [ ]:
def print_captured_inputs_summary():
    """Pretty-print summary of all captured inputs."""
    print("="*70)
    print(f"CAPTURED INPUTS SUMMARY - Layer {captured_inputs['layer_idx']}")
    print("="*70)
    print(f"\nScenario: {captured_inputs['scenario']}")
    print(f"\n{'Parameter':<25} {'Type':<20} {'Shape/Value':<25}")
    print("-"*70)
    
    for key, value in captured_inputs.items():
        if key in ['scenario', 'layer_idx']:
            continue
            
        if value is None:
            print(f"{key:<25} {'None':<20} {'-':<25}")
        elif isinstance(value, torch.Tensor):
            print(f"{key:<25} {'Tensor':<20} {str(value.shape):<25}")
        elif isinstance(value, list):
            print(f"{key:<25} {'list':<20} {f'len={len(value)}, {value}':<25}")
        elif isinstance(value, tuple):
            if all(isinstance(x, torch.Tensor) for x in value):
                shapes = tuple(x.shape for x in value)
                print(f"{key:<25} {'tuple[Tensor]':<20} {str(shapes):<25}")
            else:
                print(f"{key:<25} {'tuple':<20} {str(value):<25}")
        elif isinstance(value, dict):
            print(f"{key:<25} {'dict':<20} {f'keys={list(value.keys())}':<25}")
        elif isinstance(value, bool):
            print(f"{key:<25} {'bool':<20} {str(value):<25}")
        else:
            print(f"{key:<25} {str(type(value).__name__):<20} {str(value)[:25]:<25}")
    
    print("\n" + "="*70)
    print("UNFROZEN_IDX_OR_MASK DETAILS")
    print("="*70)
    
    unfrozen = captured_inputs['unfrozen_idx_or_mask']
    if isinstance(unfrozen, list):
        print(f"Type: list")
        print(f"Length: {len(unfrozen)}")
        print(f"Content: {unfrozen}")
        print(f"Description: List of batch indices that are still unfrozen")
    elif isinstance(unfrozen, torch.Tensor):
        print(f"Type: torch.Tensor")
        print(f"Shape: {unfrozen.shape}")
        print(f"Dtype: {unfrozen.dtype}")
        print(f"Device: {unfrozen.device}")
        print(f"Content:\n{unfrozen}")
        print(f"Description: Boolean mask [batch, generation_length] indicating which")
        print(f"             tokens are unfrozen at each timestep")
    elif unfrozen is None:
        print("Type: None")
        print("Description: All batch items are unfrozen (no early exits)")
    
    print("="*70)

# Call the helper
print_captured_inputs_summary()


In [ ]:
# Access individual captured values for testing
print("Quick access to captured values:")
print(f"- captured_inputs['hidden_states']")
print(f"- captured_inputs['attention_mask']")
print(f"- captured_inputs['unfrozen_idx_or_mask']")
print(f"- etc...")
print(f"\nExample: Call patched_layer_forward directly with captured inputs:")
print(f"  target_layer.patched_layer_forward(**{key: captured_inputs[key] for key in [...]})")

# Restore original method when done
def restore_original():
    target_layer.patched_layer_forward = original_patched_layer_forward
    print("✓ Original patched_layer_forward restored")

print("\nTo restore original: call restore_original()")


## Usage Guide

To capture inputs from `patched_layer_forward`:

1. **Run the setup cells** - defines `captured_inputs` dict and interceptor wrapper
2. **Apply monkey patch** - installs interceptor on target layer
3. **Run scenario A or B** - triggers capture during model execution
4. **Inspect results** - use `print_captured_inputs_summary()` or access `captured_inputs` directly

### What gets captured:

**Free Generate Mode (list-based unfrozen_idx_or_mask):**
- During autoregressive generation (q_len == 1)
- `unfrozen_idx_or_mask` is a list of batch indices still computing
- Example: `[0, 1, 3]` means batches 0, 1, 3 haven't exited yet

**SFT Student Mode (tensor-based unfrozen_idx_or_mask):**
- During teacher-forcing with prescribed exits
- `unfrozen_idx_or_mask` is a boolean tensor `[batch, generation_length]`
- Each `True` means that token hasn't frozen yet at this layer

### Test manually:
```python
# After capturing, you can call patched_layer_forward directly:
outputs = target_layer.patched_layer_forward(
    hidden_states=captured_inputs['hidden_states'],
    attention_mask=captured_inputs['attention_mask'],
    position_ids=captured_inputs['position_ids'],
    unfrozen_idx_or_mask=captured_inputs['unfrozen_idx_or_mask'],
    # ... other params
)
```


In [ ]:
def generate_text(model: AutoModelForCausalLM, prompt: str | List[str], system_prompt: str| List[str], prefiller: Optional[str| List[str]], tokenizer: AutoTokenizer, generation_config: dict, device: str):
    if isinstance(prompt, str):
        prompt = [prompt]
    pre_transformed_conversation = format_conversation(user_prompts = prompt, system_prompt=system_prompt)
    full_prompts = tokenizer.apply_chat_template(pre_transformed_conversation, 
                                                 prefiller=prefiller,
                                                 tokenize=False,
                                                 add_generation_prompt=True # adds the <｜Assistant｜> token at the end
                                                 )
    inputs = full_tokenize(prompts=full_prompts, tokenizer=tokenizer, device = device, add_special_tokens=False)
    print('prompt tokens shape:', inputs['input_ids'].shape)
    all_model_outputs = model.generate(**inputs, **generation_config)
    decoded_responses = tokenizer.decode(all_model_outputs[0].squeeze())
    return decoded_responses, all_model_outputs

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotary_emb): Qw

In [ ]:
def generate_text(model: AutoModelForCausalLM, prompt: str | List[str], system_prompt: str| List[str], prefiller: Optional[str| List[str]], tokenizer: AutoTokenizer, generation_config: dict, device: str):
    if isinstance(prompt, str):
        prompt = [prompt]
    pre_transformed_conversation = format_conversation(user_prompts = prompt, system_prompt=system_prompt)
    full_prompts = tokenizer.apply_chat_template(pre_transformed_conversation, 
                                                 prefiller=prefiller,
                                                 tokenize=False,
                                                 add_generation_prompt=True # adds the <｜Assistant｜> token at the end
                                                 )
    inputs = full_tokenize(prompts=full_prompts, tokenizer=tokenizer, device = device, add_special_tokens=False)
    print('prompt tokens shape:', inputs['input_ids'].shape)
    all_model_outputs = model.generate(**inputs, **generation_config)
    decoded_responses = tokenizer.decode(all_model_outputs[0].squeeze())
    return decoded_responses, all_model_outputs
